# LAB 3 — Ensaio do arco: uma notícia contra os dados do ISP

**Disciplina** Tomada de Decisão Baseada em Dados para Gestores de Segurança Pública (CAES-PMERJ / FGV)
**Aula** 3 — Estudo assíncrono no eClass (11 de novembro de 2026, horário livre)
**Duração do lab** cerca de 2 h, em grupo, depois das leituras
**Entrega** Post do grupo no fórum do eClass até **11/11/2026, 23:59**, com o texto gerado na Parte 6

> **Para quem é este lab:** gestores que decidem. Não é preciso programar. O grupo troca `'TODO'` pela resposta, entre aspas simples, e roda as células prontas (Shift+Enter). Cada resposta vira uma tabela `gold.lab3_...` no espaço `blue-blood`.

## Antes de começar

1. Importe este arquivo no Databricks Free Edition: **Workspace → (seu usuário) → Import → File → `lab-3.ipynb`**.
2. Conecte ao compute *Serverless* e rode as células na ordem, de cima para baixo. A primeira célula de código abre o espaço `blue-blood` (ou cria, se este workspace ainda não tiver).
3. Combinem um horário para o grupo trabalhar junto (videochamada): uma pessoa compartilha a tela e digita.
4. Tragam do lab 1 a CISP, o delito e os critérios do grupo. Se o lab 1 foi feito neste workspace, eles estão em `gold.lab1_decisao`, `gold.lab1_criterios` e `gold.lab1_julgamentos`. Os critérios podem ser ajustados aqui, desde que o grupo registre o porquê.

**O cenário.** A **Blue Blood** é uma força policial fictícia cuja área coincide com o estado do Rio de Janeiro: Comando de Área = RISP, batalhão = AISP, companhia = CISP. Organização, pessoas e números internos são fictícios; os dados de criminalidade são reais e públicos (ISP-RJ).

**Este lab é o ensaio geral da aula 4.** O grupo percorre as cinco etapas do arco sobre **uma** notícia real. Na aula 4, o mesmo caminho é feito com a base completa do ISP Microdados.

| Etapa do arco | Neste lab |
| --- | --- |
| 1. Pressão por decisão | A notícia e a decisão que ela pede (Parte 1) |
| 2. Critérios de decisão | Critérios, pesos e limiares, antes de ver os dados (Parte 2) |
| 3. Coleta e armazenamento | Série pública do ISPDados e teste de acesso ao ISP Microdados (Parte 3) |
| 4. Processamento e visualização | Trimestre contra o mesmo trimestre do ano anterior, em número e em taxa (Parte 4) |
| 5. Decisão com critérios e risco | Concordar ou contestar a notícia, com o risco de errar (Parte 5) |

## Objetivos de aprendizagem

Ao final deste lab, o grupo será capaz de:

1. **Transformar** uma notícia numa decisão de gestão com prazo e custo de errar.
2. **Fixar** critérios e limiares **antes** de olhar os dados.
3. **Obter** uma série oficial do ISP e registrar de onde ela veio.
4. **Comparar** períodos equivalentes, em número absoluto e em taxa por 100 mil habitantes.
5. **Concluir** se a notícia se sustenta, declarando o risco de estar errado.

## Parte 1 — A notícia e a decisão (20 min)

Escolham **uma** notícia sobre criminalidade no Rio de Janeiro que traga uma imagem negativa de alguma esfera da segurança pública (polícia, governo, um bairro, um tipo de crime). Procurem no Google News.

Boas candidatas têm:

- um **número** ou uma tendência afirmada ("disparou", "caiu 30%", "recorde");
- um **lugar** que dê para ligar a uma CISP, AISP, RISP ou município;
- um **período** identificável.

O Chefe de Comunicação Social traz as opções; o grupo escolhe.

A primeira célula abre o espaço `blue-blood`; a segunda grava a notícia e a decisão em `gold.lab3_noticia` e `gold.lab3_decisao`.

In [0]:
%sql
-- 🟢 PRONTA — cria (ou reabre) o espaço persistente da Blue Blood. Só execute.
-- Catálogo = onde ficam as tabelas. Zonas (schemas) = Bronze, Silver e Gold. Volume = onde ficam os arquivos.
-- Se reconectar o notebook, rode esta célula de novo: ela não apaga nada.
CREATE CATALOG IF NOT EXISTS `blue-blood` COMMENT 'Blue Blood (CAES-PMERJ / FGV): tabelas e arquivos dos labs';
USE CATALOG `blue-blood`;
CREATE SCHEMA IF NOT EXISTS bronze COMMENT 'Bronze: o dado como chegou, sem alteração';
CREATE SCHEMA IF NOT EXISTS silver COMMENT 'Silver: limpo, tipado e checado';
CREATE SCHEMA IF NOT EXISTS gold COMMENT 'Gold: indicadores e as decisões e parâmetros do grupo';
CREATE VOLUME IF NOT EXISTS bronze.arquivos COMMENT 'Arquivos baixados como chegaram (ISPDados, narrativas)';
SHOW SCHEMAS

In [0]:
%sql
-- Parte 1 — a notícia escolhida pelo grupo e a decisão que ela pede. Texto ENTRE ASPAS SIMPLES.
CREATE OR REPLACE TABLE gold.lab3_noticia COMMENT 'Lab 3: a notícia do ensaio e o número que ela afirma' AS
SELECT
  0 AS grupo,                      -- TODO: número do grupo, de 1 a 8 (sem aspas)
  'TODO' AS titulo,
  'TODO' AS veiculo,
  'TODO' AS data,                  -- ex.: '2026-10-28'
  'TODO' AS url,
  'TODO' AS afirmacao,             -- a frase com o número, copiada da notícia
  'TODO' AS esfera_atingida,       -- ex.: 'polícia militar', 'governo do estado', 'bairro X'
  CAST(NULL AS DOUBLE) AS variacao_afirmada_pct;   -- TODO: troque NULL pela variação afirmada, em %, ex. 40; deixe NULL se não houver número

CREATE OR REPLACE TABLE gold.lab3_decisao COMMENT 'Lab 3: a decisão de gestão que a notícia pede' AS
SELECT
  'TODO' AS decisao_em_jogo,       -- que decisão de gestão a notícia está pedindo?
  'TODO' AS quem_decide,
  'TODO' AS prazo,
  'TODO' AS custo_de_errar

---

## Parte 2 — Critérios, pesos e limiares, antes dos dados (20 min)

Reaproveitem os critérios do lab 1 ou ajustem para esta notícia. A célula vem com o exemplo da aula 1. Para dizer que o segundo importa mais, escrevam a fração, por exemplo `1/3`.

**Escala de Saaty:** 1 igual · 3 moderada · 5 forte · 7 muito forte · 9 extrema · 2, 4, 6, 8 intermediários · 1/3, 1/5... quando o segundo importa mais.

In [0]:
%sql
-- Parte 2 — critérios e julgamentos par a par (exemplo da aula 1; troque pelos do grupo).
-- Os critérios do grupo (2 a 5). Troque os nomes, se quiser, e use os MESMOS nomes nos julgamentos.
CREATE OR REPLACE TABLE gold.lab3_criterios COMMENT 'Critérios de decisão do grupo' AS
SELECT * FROM VALUES
  ('Evidência'),
  ('Custo operacional'),
  ('Legitimidade')
AS t(criterio);

-- Julgamentos par a par na escala de Saaty: quanto o primeiro importa mais que o segundo.
CREATE OR REPLACE TABLE gold.lab3_julgamentos COMMENT 'Julgamentos par a par (escala de Saaty)' AS
SELECT criterio_a, criterio_b, CAST(valor AS DOUBLE) AS valor FROM VALUES
  ('Evidência', 'Custo operacional', 3),     -- Evidência importa 3x mais que Custo
  ('Evidência', 'Legitimidade', 2),
  ('Legitimidade', 'Custo operacional', 2)       -- o par pode vir em qualquer ordem; 1/3 vale
AS t(criterio_a, criterio_b, valor);

SELECT * FROM gold.lab3_julgamentos

In [0]:
%sql
-- 🟢 PRONTA — pesos AHP (média geométrica) e razão de consistência (CR), em SQL. Só execute.
-- Resultado salvo em gold.lab3_pesos.
CREATE OR REPLACE TABLE gold.lab3_pesos COMMENT 'Pesos AHP dos critérios do grupo e razão de consistência (CR)' AS
WITH c AS (SELECT criterio FROM gold.lab3_criterios),
matriz AS (                       -- matriz par a par completa: diagonal 1, julgamento e seu inverso
  SELECT l.criterio AS linha, k.criterio AS coluna,
         CASE WHEN l.criterio = k.criterio THEN 1.0 ELSE coalesce(j.valor, 1.0 / r.valor) END AS valor
  FROM c l CROSS JOIN c k
  LEFT JOIN gold.lab3_julgamentos j ON j.criterio_a = l.criterio AND j.criterio_b = k.criterio
  LEFT JOIN gold.lab3_julgamentos r ON r.criterio_a = k.criterio AND r.criterio_b = l.criterio
),
geo AS (SELECT linha AS criterio, exp(avg(ln(valor))) AS g FROM matriz GROUP BY linha),
pesos AS (SELECT criterio, g / sum(g) OVER () AS peso FROM geo),
consistencia AS (                 -- lambda máximo e CR = ((lambda - n) / (n - 1)) / índice aleatório de Saaty
  SELECT count(*) AS n, avg(aw / peso) AS lambda_max
  FROM (SELECT m.linha, sum(m.valor * p.peso) AS aw FROM matriz m JOIN pesos p ON p.criterio = m.coluna GROUP BY m.linha) x
  JOIN pesos p ON p.criterio = x.linha
)
SELECT p.criterio, round(p.peso, 3) AS peso,
       round(CASE WHEN n < 3 THEN 0.0
                  ELSE ((lambda_max - n) / (n - 1)) / CASE n WHEN 3 THEN 0.58 WHEN 4 THEN 0.90 ELSE 1.12 END END, 3) AS cr,
       (SELECT count_if(valor IS NULL) FROM matriz) AS julgamentos_faltando
FROM pesos p CROSS JOIN consistencia;

SELECT criterio, concat(CAST(round(peso * 100) AS INT), '%') AS peso, cr,
       CASE WHEN julgamentos_faltando > 0 THEN 'FALTAM julgamentos: confira se todo par de critérios tem um valor'
            WHEN cr < 0.10 THEN 'Julgamentos coerentes: pode usar os pesos'
            ELSE 'CR acima de 0,10: os julgamentos se contradizem. Revejam o par mais discutido e rodem de novo' END AS leitura
FROM gold.lab3_pesos ORDER BY peso DESC

**O limiar da Evidência vira número.** Escrevam agora, antes de ver a série, a partir de que variação o grupo considera que **há alta real** e a partir de que diferença o grupo diz que a notícia **exagera**. Os outros critérios ficam em texto.

In [0]:
%sql
-- Parte 2 — limiares escritos ANTES de ver os dados.
CREATE OR REPLACE TABLE gold.lab3_limiares COMMENT 'Lab 3: limiares numéricos do critério Evidência, fixados antes dos dados' AS
SELECT
  15 AS limiar_alta_real_pct,       -- TODO: acima desta variação (%) contra o mesmo trimestre do ano anterior, há alta real
  10 AS tolerancia_noticia_pp;      -- TODO: diferença (pontos percentuais) aceita entre o número da notícia e o do ISP

-- Indicador e limiar de cada critério (mesmos nomes de gold.lab3_criterios).
-- A linha da Evidência já vem pronta, com o limiar numérico de cima.
CREATE OR REPLACE TABLE gold.lab3_indicadores_limiares COMMENT 'Lab 3: indicador e limiar de cada critério' AS
SELECT 'Evidência' AS criterio,
       'Variação do delito no trimestre contra o mesmo trimestre do ano anterior, na CISP (ISP)' AS indicador,
       concat('Alta real se passar de ', limiar_alta_real_pct, '%') AS limiar
FROM gold.lab3_limiares
UNION ALL
SELECT * FROM VALUES
  ('Custo operacional', 'TODO', 'TODO'),
  ('Legitimidade',      'TODO', 'TODO')
AS t(criterio, indicador, limiar)

---

## Parte 3 — Coleta: a série pública do ISP (20 min)

O ISPDados publica, de graça e sem cadastro, as séries mensais por delegacia (CISP) desde 2003. As células abaixo fazem o caminho do lab 1: baixar a série e a população por CISP para o volume (Bronze), registrar a data e a fonte em `bronze.registro_ingestao`, criar as tabelas Bronze e padronizar na Silver.

- **Série por CISP:** `BaseDPEvolucaoMensalCisp.csv`
- **População por CISP:** `PopulacaoEvolucaoMensalCisp.csv` (a última estimativa publicada é de julho de 2022; usamos a mais recente disponível)

**Delitos disponíveis (nome da coluna):** `roubo_rua` (soma de transeunte, celular e coletivo), `roubo_transeunte`, `roubo_celular`, `roubo_em_coletivo`, `roubo_veiculo`, `roubo_carga`, `roubo_comercio`, `total_roubos`, `furto_transeunte`, `furto_celular`, `total_furtos`, `letalidade_violenta`, `hom_doloso`, `lesao_corp_dolosa`, `estupro`, entre outros.

> Se a rede falhar, o notebook grava no volume um **plano B embutido**: dados reais do ISPDados para todas as CISPs da RISP 1 e as CISPs 33 e 64, com os 14 delitos acima, de jan/2024 a set/2026, baixados em 26/10/2026.

In [0]:
%sql
-- Parte 3 — o recorte do grupo (o mesmo do lab 1).
CREATE OR REPLACE TABLE gold.lab3_recorte COMMENT 'Lab 3: CISP e delito do ensaio' AS
SELECT
  12 AS cisp,                -- TODO: número da CISP do grupo (sem aspas)
  'roubo_rua' AS delito      -- TODO: nome da coluna do delito, da lista acima

In [0]:
# 🟢 PRONTA — baixa as séries públicas do ISPDados para o volume da Bronze. Só execute.
# O Databricks Free Edition limita o acesso à internet: se o site do ISP estiver bloqueado ou fora do ar,
# a célula grava no volume o PLANO B, uma amostra real do ISPDados embutida aqui (CISPs da RISP 1 e as
# CISPs 33 e 64, jan/2024 a set/2026, 14 delitos), baixada em 26/10/2026. Python só para o download; o resto é SQL.
import base64, gzip, urllib.request

VOLUME = "/Volumes/blue-blood/bronze/arquivos"
URLS = {
    "BaseDPEvolucaoMensalCisp.csv": "https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv",
    "PopulacaoEvolucaoMensalCisp.csv": "https://www.ispdados.rj.gov.br/Arquivos/PopulacaoEvolucaoMensalCisp.csv",
}
PLANO_B = {  # gzip + base64 dos dois arquivos, no mesmo formato do ISP (separador ;, latin-1)
    "BaseDPEvolucaoMensalCisp.csv": "...",
    "PopulacaoEvolucaoMensalCisp.csv": "H4sIAAAAAAACA03US64kRwgF0HmshUHwB9ViLOvJAw/sbtn7l3x5cpI9LApFAnGIrz//+fr89ce/n9///vH5+ePnb18IHP74R67Ih9ONnLOMjz1B1RIndb18/AnatQqkmtuJJ1gRl6mzLU5uplY7cXf76f2QhSIa0nb4bpRv4lPFVYe3qMQJSXUnVZ5gl2aT3BI+rG+q3yBJkTy8DbSLFLmmHt4GypKT3KQQzbcD1ku3Oftw7bESylSmF7m/9NDmQVUdR+4vI2xEucWObA9S7mFkrHlkmxBPDqH5mLw9pPqliEoca+9oUL5Sa/CRbYK1NZmsw4/EG42MJrs3j+65Ejd0Couj2y8G1R2kdv1ovVFtUaoWP/aWMHO95Cl6YqOWlSrkLQFHsUNQynu9x1Gso9tIxDThKNZRJIUHRhtPsPCbKfBfw1GsozTCwVFwFK8jSZr/xlGso0CumA6jWEYoivH9GEexjjBWLTUbR5sLHeQ4xMfRk+tcRlyXeSA9xaKBolsc35C2hVtN06sMpOdc8UYuLlMG0jbRIxzj7YH0znAaFsUlyHYBSAaL2hUDKRbSNUFUYyTFSmIjLBpmK/YOBxO81HM3sl2MJKyO56iLNxp5saatPZJiJWHxKYe4bsegVH2JcWtDaaNajFsvbK+9NQjgowvWb0qxlBxC4s6afXLHINgx1e8nKR9K2fBp7JCUK8kKRUEIJOVKEnznMoK5mYohIpg4svc7FrgcmKmBlC8kESxjykjKlcR4T/iyjKR8JAku0b0iR9LmGpauQkbH1g9J2lQi/S0pV5IW3sRr35K2BawBlSseVUjKlYT3BC9HDKR8IfH3NulAeieIqxpeMZByIQksjvyB9EQ9ehoG54H0fCxxtwj2ALV3OBrYPfQtAykXkhiWF3gH0kYjEO1KHki5kMAONeC9120YkLLw1P0PaaN46Zyk0Ju9NQDSFbpYioGUCwl5KBgP63+EPbVX8gYAAA==",
}

try:
    conteudos = {}
    for nome, url in URLS.items():
        with urllib.request.urlopen(url, timeout=60) as r:
            conteudos[nome] = r.read()
    ORIGEM_ISP = "ISPDados (download agora)"
except Exception as erro:
    conteudos = {nome: gzip.decompress(base64.b64decode(b)) for nome, b in PLANO_B.items()}
    ORIGEM_ISP = f"Plano B embutido (ISPDados de 26/10/2026); download falhou: {type(erro).__name__}"

spark.sql("""CREATE TABLE IF NOT EXISTS bronze.registro_ingestao
             (arquivo STRING, origem STRING, url STRING, bytes BIGINT, baixado_em TIMESTAMP)
             COMMENT 'Bronze: de onde veio cada arquivo e quando'""")
for nome, dados in conteudos.items():
    with open(f"{VOLUME}/{nome}", "wb") as f:
        f.write(dados)
    spark.sql("INSERT INTO bronze.registro_ingestao VALUES (:arquivo, :origem, :url, :bytes, current_timestamp())",
              args={"arquivo": nome, "origem": ORIGEM_ISP, "url": URLS[nome], "bytes": len(dados)})
print(ORIGEM_ISP)
print(f"Arquivos gravados em {VOLUME}: {', '.join(conteudos)}")

In [0]:
%sql
-- 🟢 PRONTA — Bronze: cada arquivo do volume vira uma tabela, sem alterar nenhum valor (tudo como texto). Só execute.
CREATE OR REPLACE TABLE bronze.isp_cisp_mensal
COMMENT 'Bronze: ISPDados BaseDPEvolucaoMensalCisp.csv como chegou (ocorrências por CISP e mês)'
AS SELECT *, current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/BaseDPEvolucaoMensalCisp.csv',
                format => 'csv', header => true, sep => ';', encoding => 'ISO-8859-1', inferColumnTypes => false);

CREATE OR REPLACE TABLE bronze.isp_populacao_cisp
COMMENT 'Bronze: ISPDados PopulacaoEvolucaoMensalCisp.csv como chegou (população estimada por CISP)'
AS SELECT *, current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/blue-blood/bronze/arquivos/PopulacaoEvolucaoMensalCisp.csv',
                format => 'csv', header => true, sep => ';', inferColumnTypes => false);

SELECT 'bronze.isp_cisp_mensal' AS tabela, count(*) AS linhas FROM bronze.isp_cisp_mensal
UNION ALL
SELECT 'bronze.isp_populacao_cisp', count(*) FROM bronze.isp_populacao_cisp

In [0]:
%sql
-- 🟢 PRONTA — Silver: uma linha por CISP, mês e delito, com tipos certos. Só execute.
-- fase 2 = mês preliminar do ISP (ainda pode mudar); fase 3 = consolidado.
CREATE OR REPLACE TABLE silver.isp_ocorrencias
COMMENT 'Silver: ocorrências por CISP, mês e delito (ISPDados), tipadas; fase 2 = preliminar'
AS
SELECT cisp, aisp, risp, munic, make_date(ano, mes, 1) AS mes_ref, ano, mes, fase, delito, ocorrencias
FROM (
  SELECT CAST(cisp AS INT) AS cisp, CAST(aisp AS INT) AS aisp, CAST(risp AS INT) AS risp, munic,
         CAST(ano AS INT) AS ano, CAST(mes AS INT) AS mes, CAST(fase AS INT) AS fase,
         CAST(roubo_rua AS INT) AS roubo_rua,
         CAST(roubo_transeunte AS INT) AS roubo_transeunte,
         CAST(roubo_celular AS INT) AS roubo_celular,
         CAST(roubo_em_coletivo AS INT) AS roubo_em_coletivo,
         CAST(roubo_veiculo AS INT) AS roubo_veiculo,
         CAST(roubo_comercio AS INT) AS roubo_comercio,
         CAST(total_roubos AS INT) AS total_roubos,
         CAST(furto_transeunte AS INT) AS furto_transeunte,
         CAST(furto_celular AS INT) AS furto_celular,
         CAST(total_furtos AS INT) AS total_furtos,
         CAST(letalidade_violenta AS INT) AS letalidade_violenta,
         CAST(hom_doloso AS INT) AS hom_doloso,
         CAST(lesao_corp_dolosa AS INT) AS lesao_corp_dolosa,
         CAST(estupro AS INT) AS estupro
  FROM bronze.isp_cisp_mensal
)
UNPIVOT (ocorrencias FOR delito IN (roubo_rua, roubo_transeunte, roubo_celular, roubo_em_coletivo, roubo_veiculo, roubo_comercio, total_roubos, furto_transeunte, furto_celular, total_furtos, letalidade_violenta, hom_doloso, lesao_corp_dolosa, estupro))
WHERE cisp IS NOT NULL AND mes BETWEEN 1 AND 12;

CREATE OR REPLACE TABLE silver.isp_populacao
COMMENT 'Silver: última população estimada pelo ISP para cada CISP (a série vai até jul/2022)'
AS
SELECT CAST(circ AS INT) AS cisp,
       max_by(CAST(replace(pop_circ, ',', '.') AS DOUBLE), CAST(ano AS INT) * 100 + CAST(mes AS INT)) AS populacao,
       max(make_date(CAST(ano AS INT), CAST(mes AS INT), 1)) AS referencia
FROM bronze.isp_populacao_cisp
WHERE trim(pop_circ) <> ''
GROUP BY CAST(circ AS INT);

SELECT count(DISTINCT cisp) AS cisps, count(DISTINCT delito) AS delitos,
       min(mes_ref) AS primeiro_mes, max(mes_ref) AS ultimo_mes,
       count_if(fase = 2) AS linhas_preliminares
FROM silver.isp_ocorrencias

In [0]:
%sql
-- 🟢 PRONTA — confere o recorte e registra a fonte do ensaio. Só execute.
CREATE OR REPLACE TABLE gold.lab3_fonte COMMENT 'Lab 3: de onde veio a série do ensaio e até quando vai' AS
SELECT r.cisp, r.delito,
       (SELECT max_by(risp, mes_ref) FROM silver.isp_ocorrencias o WHERE o.cisp = r.cisp) AS risp,
       (SELECT max(mes_ref) FROM silver.isp_ocorrencias o WHERE o.cisp = r.cisp AND o.delito = r.delito) AS ultimo_mes,
       (SELECT max_by(origem, baixado_em) FROM bronze.registro_ingestao WHERE arquivo = 'BaseDPEvolucaoMensalCisp.csv') AS origem,
       (SELECT max(baixado_em) FROM bronze.registro_ingestao WHERE arquivo = 'BaseDPEvolucaoMensalCisp.csv') AS baixado_em
FROM gold.lab3_recorte r;

SELECT *, CASE WHEN ultimo_mes IS NULL
               THEN 'ATENÇÃO: esta CISP ou este delito não estão nos dados carregados. Confira o número e o nome da coluna (no plano B só há as CISPs da RISP 1, 33 e 64).'
               ELSE 'Recorte encontrado.' END AS conferencia
FROM gold.lab3_fonte

### Atividade 3.2: teste de acesso ao ISP Microdados (10 min)

Na aula 4, **cada aluno** baixa a base do [ISP Microdados](https://ispconecta.rj.gov.br/microdados/) com as próprias credenciais. Hoje, cada integrante só **testa o acesso**: abre a ferramenta, vê o formulário e os filtros, e **não baixa a base final**.

> **Portaria ISP 127/2022:** dados fora dos sites públicos do ISP são fornecidos mediante Termo de Responsabilidade; é proibido comercializar ou repassar a base a terceiros; o uso fica restrito a estudos próprios; inconsistências devem ser comunicadas ao ISP. Por isso o professor não distribui a base e cada um baixa a sua.

A Corregedora e Governança de Dados confere que todos registraram o teste.

In [0]:
%sql
-- Atividade 3.2 — um registro por integrante. Não baixe a base final hoje.
-- abriu_o_formulario: 'sim' ou 'não'. Se 'não', descreva o problema em observacao.
CREATE OR REPLACE TABLE gold.lab3_teste_microdados COMMENT 'Lab 3: teste de acesso ao ISP Microdados, por integrante' AS
SELECT * FROM VALUES
  ('Comandante-Geral',                         'TODO', 'TODO', ''),
  ('Chefe de Inteligência e Análise Criminal', 'TODO', 'TODO', ''),
  ('Chefe de Comunicação Social',              'TODO', 'TODO', ''),
  ('Corregedora e Governança de Dados',        'TODO', 'TODO', ''),
  ('Comandante de Área Operacional',           'TODO', 'TODO', '')
AS t(papel, data, abriu_o_formulario, observacao)

---

## Parte 4 — Processamento: compare períodos equivalentes (20 min)

A célula SQL abaixo monta a série da CISP e da RISP na Gold (`gold.lab3_serie`) e calcula, para o delito e a CISP do grupo (`gold.lab3_comparacao`):

- o **último trimestre** disponível contra o **mesmo trimestre do ano anterior** (tira o efeito da época do ano);
- o último trimestre contra o **trimestre imediatamente anterior** (o que muitas manchetes comparam);
- a **taxa por 100 mil habitantes**, na CISP e na RISP inteira, para saber se a CISP está acima ou abaixo da sua região.

**Armadilhas que a célula aponta:** base pequena (poucos casos fazem qualquer variação parecer enorme) e dados preliminares do ISP (`fase 2`, ainda sujeitos a revisão).

In [0]:
%sql
-- 🟢 PRONTA — Gold: série da CISP e da RISP, trimestres comparáveis e taxa por 100 mil. Só execute.
CREATE OR REPLACE TABLE gold.lab3_serie COMMENT 'Lab 3: série mensal do delito na CISP do grupo e na sua RISP, com população' AS
WITH f AS (SELECT * FROM gold.lab3_fonte)
SELECT 'CISP' AS nivel, o.mes_ref, o.fase, o.ocorrencias AS casos, p.populacao
FROM silver.isp_ocorrencias o JOIN f ON o.cisp = f.cisp AND o.delito = f.delito
LEFT JOIN silver.isp_populacao p ON p.cisp = o.cisp
UNION ALL
SELECT 'RISP', o.mes_ref, min(o.fase), sum(o.ocorrencias),
       (SELECT sum(p.populacao) FROM silver.isp_populacao p
        WHERE p.cisp IN (SELECT DISTINCT cisp FROM silver.isp_ocorrencias x WHERE x.risp = (SELECT risp FROM f)))
FROM silver.isp_ocorrencias o JOIN f ON o.risp = f.risp AND o.delito = f.delito
GROUP BY o.mes_ref;

CREATE OR REPLACE TABLE gold.lab3_comparacao COMMENT 'Lab 3: último trimestre contra o mesmo do ano anterior e contra o anterior, em número e taxa' AS
WITH u AS (SELECT max(mes_ref) AS fim FROM gold.lab3_serie WHERE nivel = 'CISP'),
t AS (
  SELECT nivel, max(u.fim) AS fim,
         sum(CASE WHEN mes_ref BETWEEN add_months(fim, -2) AND fim THEN casos END) AS atual,
         sum(CASE WHEN mes_ref BETWEEN add_months(fim, -14) AND add_months(fim, -12) THEN casos END) AS ano_anterior,
         sum(CASE WHEN mes_ref BETWEEN add_months(fim, -5) AND add_months(fim, -3) THEN casos END) AS tri_anterior,
         count_if(mes_ref BETWEEN add_months(fim, -2) AND fim AND fase = 2) AS meses_preliminares,
         max(populacao) AS populacao
  FROM gold.lab3_serie CROSS JOIN u GROUP BY nivel
)
SELECT nivel,
       concat(date_format(add_months(fim, -2), 'MM/yyyy'), ' a ', date_format(fim, 'MM/yyyy')) AS ultimo_trimestre,
       atual, ano_anterior, tri_anterior,
       round(try_divide(atual - ano_anterior, ano_anterior) * 100, 1) AS var_ano_pct,
       round(try_divide(atual - tri_anterior, tri_anterior) * 100, 1) AS var_tri_pct,
       round(atual / populacao * 100000, 1) AS taxa_100mil_trimestre,
       round(ano_anterior / populacao * 100000, 1) AS taxa_100mil_ano_anterior,
       concat_ws(' | ',
         CASE WHEN least(atual, ano_anterior) < 30
              THEN concat('Base pequena: ', ano_anterior, ' casos no trimestre de comparação; variações percentuais oscilam muito') END,
         CASE WHEN meses_preliminares > 0 THEN 'O último trimestre tem dados preliminares do ISP (fase 2), sujeitos a revisão' END,
         'População: estimativa mais recente do ISP por CISP (até julho de 2022)') AS avisos
FROM t;

SELECT * FROM gold.lab3_comparacao ORDER BY nivel

In [0]:
# 🟢 PRONTA — gráfico da CISP e da RISP, lido da Gold. Só execute.
import matplotlib.pyplot as plt

serie = spark.table("gold.lab3_serie").orderBy("mes_ref").toPandas()
fonte = spark.table("gold.lab3_fonte").first()
if serie.empty:
    raise ValueError("A série está vazia: confira a CISP e o delito na Parte 3.")
cisp = serie[serie["nivel"] == "CISP"].tail(24).reset_index(drop=True)
risp = serie[serie["nivel"] == "RISP"].set_index("mes_ref").reindex(cisp["mes_ref"]).reset_index()
rot = cisp["mes_ref"].astype(str).str[:7]
n = len(cisp)
cores = ["#1d6fb8" if i >= n - 3 else ("#8a96a3" if n - 15 <= i < n - 12 else "#c9d3dd") for i in range(n)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.bar(rot, cisp["casos"], color=cores)
ax1.set_title(f"CISP {fonte['cisp']} · {fonte['delito']} · casos por mês")
ax2.plot(rot, cisp["casos"] / cisp["populacao"] * 1e5, marker="o", color="#0b2a4a", label=f"CISP {fonte['cisp']}")
ax2.plot(rot, risp["casos"] / risp["populacao"] * 1e5, linestyle="--", color="#8a96a3", label=f"RISP {fonte['risp']}")
ax2.set_title("Taxa por 100 mil habitantes, por mês")
ax2.legend()
for ax in (ax1, ax2):
    ax.tick_params(axis="x", rotation=90, labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Azul: último trimestre · cinza escuro: mesmo trimestre do ano anterior", fontsize=10)
plt.tight_layout()
plt.show()

---

## Parte 5 — Decisão com critérios e risco (20 min)

A célula abaixo aplica os limiares da tabela `gold.lab3_limiares`, que o grupo escreveu **antes** de ver os dados, e grava a leitura em `gold.lab3_leitura`. Ela sugere uma leitura. A posição final é do grupo, que pode discordar da sugestão, desde que explique por quê.

In [0]:
%sql
-- 🟢 PRONTA — aplica os limiares da Parte 2. Só execute.
CREATE OR REPLACE TABLE gold.lab3_leitura COMMENT 'Lab 3: leitura sugerida pela regra do grupo' AS
SELECT coalesce(CAST(n.variacao_afirmada_pct AS STRING), 'sem número') AS noticia_afirma_pct,
       c.var_ano_pct AS isp_mesmo_trimestre_ano_anterior_pct,
       c.var_tri_pct AS isp_trimestre_anterior_pct,
       CASE WHEN n.variacao_afirmada_pct IS NULL
              THEN 'Sem número na notícia: compare a tendência afirmada com a variação do ISP.'
            WHEN abs(n.variacao_afirmada_pct - c.var_ano_pct) <= l.tolerancia_noticia_pp
              THEN 'O ISP CONFIRMA a ordem de grandeza da notícia.'
            WHEN abs(n.variacao_afirmada_pct - c.var_tri_pct) <= l.tolerancia_noticia_pp
              THEN 'O número da notícia bate com a comparação contra o trimestre ANTERIOR, que mistura época do ano. Contra o mesmo trimestre do ano anterior, o quadro é outro.'
            ELSE 'O ISP CONTESTA a notícia: a variação oficial é bem diferente da afirmada.' END AS leitura_sugerida,
       CASE WHEN c.var_ano_pct > l.limiar_alta_real_pct THEN 'SIM' ELSE 'NÃO' END AS alta_real_pelo_limiar,
       l.limiar_alta_real_pct
FROM gold.lab3_noticia n CROSS JOIN gold.lab3_limiares l
JOIN gold.lab3_comparacao c ON c.nivel = 'CISP';

SELECT * FROM gold.lab3_leitura

In [0]:
%sql
-- Parte 5 — a posição do grupo.
CREATE OR REPLACE TABLE gold.lab3_posicao COMMENT 'Lab 3: posição do grupo, decisão, risco de errar e acompanhamento' AS
SELECT
  'TODO' AS posicao,                      -- 'concorda', 'contesta' ou 'inconclusivo'
  'TODO' AS decisao_final,                -- o que o comando faz agora (uma ou duas frases)
  'TODO' AS risco_de_errar,               -- o que pode estar errado na conclusão e o que acontece se estiver
  'TODO' AS indicador_de_acompanhamento   -- que número o grupo acompanha nos próximos meses

---

## Parte 6 — Rodada 3 da Operação Manchete e o post no fórum (10 min)

**Rodada 3 — O atalho.** Um analista quer mandar a base do ISP Microdados por e-mail aos batalhões e a um consultor externo "para ganhar tempo".

| Porta | Escolha |
| --- | --- |
| A | Autorizar o envio: "é dado público" |
| B | Cada unidade baixa a base com a própria credencial e registra data, filtros e o Termo de Responsabilidade |
| C | Batalhões usam só as séries públicas do ISPDados; microdados ficam na Inteligência |
| D | Proibir o uso de microdados no comando |
| E | Publicar a base tratada no site da Blue Blood |

O grupo escolhe uma porta e justifica em uma frase. As consequências e os pontos são revelados pelo professor na abertura da aula 4.

In [0]:
%sql
-- Parte 6 — Rodada 3.
CREATE OR REPLACE TABLE gold.lab3_rodada3 COMMENT 'Lab 3: porta escolhida na Rodada 3 da Operação Manchete' AS
SELECT
  'TODO' AS porta,           -- 'A', 'B', 'C', 'D' ou 'E'
  'TODO' AS justificativa    -- uma frase

### Checagem final e texto do fórum

Rode as duas células. A primeira lista o que falta; a segunda monta, a partir das tabelas do grupo, o texto para copiar no fórum do eClass.

In [0]:
%sql
-- 🟢 PRONTA — confere o que falta. Só execute.
-- Se aparecer TABLE_OR_VIEW_NOT_FOUND, a célula daquela parte ainda não foi executada.
SELECT item, CASE WHEN coalesce(ok, false) THEN 'OK' ELSE 'FALTA' END AS situacao
FROM (
  SELECT 1, 'Parte 1 — número do grupo (1 a 8)', (SELECT max(grupo) BETWEEN 1 AND 8 FROM gold.lab3_noticia)
  UNION ALL SELECT 2, 'Parte 1 — notícia completa', (SELECT count_if(to_json(named_struct('t', titulo, 'v', veiculo, 'd', data, 'u', url, 'a', afirmacao, 'e', esfera_atingida)) LIKE '%TODO%') = 0 FROM gold.lab3_noticia)
  UNION ALL SELECT 3, 'Parte 1 — decisão nomeada', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab3_decisao)
  UNION ALL SELECT 4, 'Parte 2 — julgamentos completos e coerentes (CR < 0,10)', (SELECT max(cr) < 0.10 AND max(julgamentos_faltando) = 0 FROM gold.lab3_pesos)
  UNION ALL SELECT 5, 'Parte 2 — indicador e limiar para cada critério', (SELECT count(i.criterio) = count(*) AND count_if(i.indicador = 'TODO' OR i.limiar = 'TODO') = 0 FROM gold.lab3_criterios c LEFT JOIN gold.lab3_indicadores_limiares i ON i.criterio = c.criterio)
  UNION ALL SELECT 6, 'Parte 3 — série do ISP carregada da rede (não do plano B)', (SELECT max(origem LIKE 'ISPDados%' AND ultimo_mes IS NOT NULL) FROM gold.lab3_fonte)
  UNION ALL SELECT 7, 'Parte 3 — teste do ISP Microdados registrado pelos 5', (SELECT count_if(data = 'TODO' OR abriu_o_formulario NOT IN ('sim', 'não')) = 0 FROM gold.lab3_teste_microdados)
  UNION ALL SELECT 8, 'Parte 5 — posição do grupo', (SELECT max(posicao IN ('concorda', 'contesta', 'inconclusivo')) FROM gold.lab3_posicao)
  UNION ALL SELECT 9, 'Parte 5 — decisão, risco e acompanhamento', (SELECT count(*) > 0 AND count_if(to_json(struct(*)) LIKE '%TODO%') = 0 FROM gold.lab3_posicao)
  UNION ALL SELECT 10, 'Parte 6 — porta e justificativa da Rodada 3', (SELECT max(porta IN ('A', 'B', 'C', 'D', 'E') AND justificativa NOT IN ('TODO', '')) FROM gold.lab3_rodada3)
) AS t(ordem, item, ok)
ORDER BY ordem

In [0]:
%sql
-- 🟢 PRONTA — texto do fórum, lido das tabelas do grupo. Copie a coluna "texto", de cima para baixo.
WITH n AS (SELECT * FROM gold.lab3_noticia), d AS (SELECT * FROM gold.lab3_decisao),
     f AS (SELECT * FROM gold.lab3_fonte), p AS (SELECT * FROM gold.lab3_posicao),
     c AS (SELECT * FROM gold.lab3_comparacao WHERE nivel = 'CISP'), r AS (SELECT * FROM gold.lab3_comparacao WHERE nivel = 'RISP')
SELECT texto FROM (
  SELECT 1 AS o, concat('Grupo ', n.grupo, ' — Ensaio do arco (lab 3)') AS texto FROM n
  UNION ALL SELECT 2, concat('1. Pressão: "', n.titulo, '" (', n.veiculo, ', ', n.data, ') ', n.url) FROM n
  UNION ALL SELECT 3, concat('   Afirmação: ', n.afirmacao) FROM n
  UNION ALL SELECT 4, concat('   Decisão em jogo: ', d.decisao_em_jogo, ' · decide: ', d.quem_decide, ' · prazo: ', d.prazo) FROM d
  UNION ALL SELECT 5, concat('2. Critérios (CR = ', max(w.cr), '): ',
                             concat_ws('; ', collect_list(concat(w.criterio, ' (', CAST(round(w.peso * 100) AS INT), '%): ', i.limiar))))
            FROM gold.lab3_pesos w LEFT JOIN gold.lab3_indicadores_limiares i ON i.criterio = w.criterio
  UNION ALL SELECT 6, concat('3. Dados: ISP, CISP ', f.cisp, ', ', f.delito, ', até ', date_format(f.ultimo_mes, 'MM/yyyy'),
                             ' (', f.origem, '; baixado em ', date_format(f.baixado_em, 'dd/MM/yyyy HH:mm'), ').') FROM f
  UNION ALL SELECT 7, '   Teste de acesso ao ISP Microdados registrado pelos 5 integrantes.'
  UNION ALL SELECT 8, concat('4. Resultado: ', c.var_ano_pct, '% contra o mesmo trimestre do ano anterior; ', c.var_tri_pct,
                             '% contra o trimestre anterior; taxa de ', c.taxa_100mil_trimestre, ' por 100 mil na CISP contra ',
                             r.taxa_100mil_trimestre, ' na RISP ', f.risp, '.') FROM c CROSS JOIN r CROSS JOIN f
  UNION ALL SELECT 9, concat('5. Posição: ', p.posicao, '. Decisão: ', p.decisao_final) FROM p
  UNION ALL SELECT 10, concat('   Risco de errar: ', p.risco_de_errar) FROM p
  UNION ALL SELECT 11, concat('   Acompanhamento: ', p.indicador_de_acompanhamento) FROM p
  UNION ALL SELECT 12, concat('Rodada 3: porta ', porta, ' — ', justificativa) FROM gold.lab3_rodada3
) ORDER BY o

## Material de apoio

- **Roteiro da aula 3** (slides no eClass): leituras, os cinco passos do ensaio e o modelo do post.
- [ISPDados — dados abertos](https://www.ispdados.rj.gov.br/estatistica.html): séries mensais por CISP e população estimada.
- [ISP Microdados](https://ispconecta.rj.gov.br/microdados/) e [Portaria ISP nº 127/2022](http://arquivo.proderj.rj.gov.br/isp_imagens/Uploads/PortariaISP127_Fornecimento_Microdados.pdf).
- **No espaço `blue-blood`:** a série padronizada (`silver.isp_ocorrencias`), a comparação (`gold.lab3_comparacao`) e todas as tabelas `gold.lab3_` com as escolhas do grupo ficam guardadas para a aula 4.
- INSTITUTO SOU DA PAZ. [Segurança Pública em Dados: Guia Prático para Jornalistas](https://lp.soudapaz.org/guiajornalistas) — como checar um número de segurança.

**Na aula 4 (12/11, presencial):** o arco completo com a base do ISP Microdados, NLP sobre as narrativas, dashboard, decisão e proposta de política pública; apresentação dos grupos à tarde.